# SelfSupervised-TTT-ImageClassification on Google Colab

This notebook sets up a Colab GPU runtime, installs dependencies, runs the Stage A SimCLR pipeline, and shows where to find logs and checkpoints.

Recommended first run:
- Use a GPU runtime in Colab.
- Keep `simclr.epochs: 1` in `configs/default.yaml` for a smoke test.
- After the smoke test succeeds, increase epochs and rerun the training cell.


## 1. Runtime Configuration

Before running the next cell, switch Colab to GPU:
- `Runtime -> Change runtime type -> T4 GPU` (or any available GPU)


In [1]:
REPO_URL = "https://github.com/3v3r51nc3/SelfSupervised-TTT-ImageClassification.git"
BRANCH = "main"
PROJECT_DIR = "/content/SelfSupervised-TTT-ImageClassification"
USE_GOOGLE_DRIVE = True
DRIVE_RUNS_DIR = "/content/drive/MyDrive/selfsupervised_ttt_runs"


In [2]:
import os
import shutil
import subprocess
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if Path(PROJECT_DIR).exists():
    print(f"Removing existing directory: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")


In [ ]:
!nvidia-smi || true
!python --version
#!pip install -r requirements.txt


## 2. Inspect or Modify the Config

The default config already points to local relative folders that work in Colab:
- `./data`
- `./logs`
- `./checkpoints`

For a first Colab run, keep `simclr.epochs: 1`.


In [4]:
!sed -n '1,220p' configs/default.yaml


In [5]:
# Optional helper: force a 1-epoch smoke test inside Colab.
# Leave this cell commented out if your config is already correct.
# !python - <<'PY'
# from pathlib import Path
# path = Path('configs/default.yaml')
# text = path.read_text()
# text = text.replace('epochs: 1', 'epochs: 1', 1)
# path.write_text(text)
# print(path.read_text())
# PY


## 3. Run Stage A Training


In [6]:
!python main.py --config configs/default.yaml


## 4. Inspect Outputs


In [ ]:
!ls -R logs
!echo
!ls -R checkpoints


In [ ]:
!sed -n '1,240p' logs/simclr-vit-cifar10-ter/experiment.log


In [ ]:
import pandas as pd

metrics_path = Path('logs/simclr-vit-cifar10-ter/metrics.csv')
if metrics_path.exists():
    df = pd.read_csv(metrics_path, header=None, names=['step', 'tag', 'value'])
    display(df)
else:
    print(f'Metrics file not found: {metrics_path}')


## 5. TensorBoard

If the `tensorboard` package is installed successfully, you can inspect the run directly in Colab.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs


## 6. Persist Artifacts to Google Drive

Colab local storage is temporary. If `USE_GOOGLE_DRIVE = True`, run the next cell after training to save outputs.


In [ ]:
if USE_GOOGLE_DRIVE:
    runs_dir = Path(DRIVE_RUNS_DIR)
    runs_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree('logs', runs_dir / 'logs', dirs_exist_ok=True)
    shutil.copytree('checkpoints', runs_dir / 'checkpoints', dirs_exist_ok=True)
    print(f'Saved logs and checkpoints to {runs_dir}')
else:
    print('Set USE_GOOGLE_DRIVE = True at the top of the notebook to enable Drive persistence.')
